# 🛒 Multi-Vendor Market Platform: Arxitektura va Fayllar Tahlili (`takrorlash.ipynb`)

Ushbu Jupyter Notebook **Multi-Vendor Market (Uzum / Ozon analogi)** loyihasining to'liq tahlilini, tizimda jarayonlarda qatnashuvchi **barcha fayllarning ishlash mexanizmlari**, ma'lumotlar oqimi (data flow), xavfsizlik va integratsiyalarini o'z ichiga oladi.

---

## 📌 Loyiha Arxitekturasi va Texnologik Stek

- **Backend Framework**: `FastAPI` (Python 3.12+, asinxron RESTful API)
- **Ma'lumotlar Bazasi & ORM**: `PostgreSQL 16` + `SQLAlchemy 2.0` (Declarative Mapped modellar) + `Alembic`
- **Xavfsizlik**: `Argon2-cffi` (parol xeshlash), `python-jose` (JWT Tokenlar: Access va Refresh), `zxcvbn` (parol mustahkamligi), `SlowAPI` (Rate Limiting)
- **Kesh & Asinxron Navbat**: `Redis 7` (OTP kodlar, kesh, tokenlar) + `Celery & Celery Beat` (Email yuborish, fon vazifalari)
- **Admin Boshqaruv**: `Starlette Admin` (SQLAlchemy modellarini vizual boshqarish va integratsiya)
- **Frontend Qatlami**: Vanilla JavaScript (`static/js/api.js`, `static/js/app.js`, `static/js/mock-data.js`) + 17 ta HTML sahifa
- **DevOps**: `Docker & Docker Compose` (5 ta konteyner: db, redis, fastapi, celery_worker, celery_beat) + `uv` paket menejeri

```mermaid
flowchart TD
    Client["Mijoz / Brauzer (/static/*.html)"] -->|HTTP Request| MW["Middleware (Session, TimeCounter, SlowAPI)"]
    MW --> FastAPIMain["FastAPI App (app/main.py)"]
    FastAPIMain --> Routers["API Routers (Auth, User, Product, Order, Common)"]
    FastAPIMain --> Admin["Starlette Admin (/admin)"]
    
    Routers --> Dependencies["Dependencies & Security (RBAC, JWT, DB Session)"]
    Dependencies --> ORM["SQLAlchemy ORM (app/model/*)"]
    Dependencies --> Utils["Utils (Argon2, Jose JWT, Redis Client)"]
    
    Routers --> CeleryApp["Celery Worker (send_email_message)"]
    CeleryApp --> RedisBroker[("Redis Broker & Cache")]
    Utils --> RedisBroker
    ORM --> PostgresDB[("PostgreSQL 16")]
```


---
## 1-BOB. Konfiguratsiya va Dastur Kirish Nuqtalari (Configuration & Entrypoints)

Tizimning asosiy sozlamalari, konteynerizatsiyasi va dastur ishga tushish zanjirini ta'minlovchi fayllar:

### 1.1. `app/config.py`
- **Vazifasi**: Loyihaning markaziy sozlamalarini boshqarish (`pydantic_settings.BaseSettings`).
- **Ishlash mexanizmi**:
  - Loyiha ildizidagi `.env` fayldan qiymatlarni o'qiydi.
  - Ma'lumotlar bazasi ulanish parametrlari (`DB_USER`, `DB_PASSWORD`, `DB_HOST`, `DB_PORT`, `DB_NAME`).
  - Xavfsizlik parametrlari (`SECRET_KEY`, `ALGORITHM = "HS256"`, `ACCESS_TOKEN_EXPIRE_MINUTES = 60`, `REFRESH_TOKEN_EXPIRE_DAYS = 1`).
  - Media konfiguratsiyasi (`MEDIA_PATH = "media"`, `FILE_SIZE = 5MB`, `FILE_TYPE = [.jpg, .png, .jpeg, .pdf, .docx, .doc, .txt]`).
  - Tashqi xizmatlar (`REDIS_URL`, `CELERY_BROKER_URL`, `CELERY_RESULT_BACKEND`, `EMAIL_FROM`, `SMTP_SERVER`, `SMTP_PORT`, `EMAIL_PASSWORD`).
  - `model_config = SettingsConfigDict(env_file=BASE_DIR / ".env", case_sensitive=True)` orqali konfiguratsiya qilinadi.

### 1.2. `app/main.py`
- **Vazifasi**: FastAPI ilovasi markazi (App Factory / Entrypoint).
- **Ishlash mexanizmi**:
  1. **SQLAlchemy & Starlette Admin Monkey Patching**:
     ```python
     if not hasattr(sqlalchemy.orm.attributes, "ScalarObjectAttributeImpl"):
         sqlalchemy.orm.attributes.ScalarObjectAttributeImpl = getattr(
             sqlalchemy.orm.attributes, "_ScalarObjectAttributeImpl", None
         )
     ```
     SQLAlchemy 2.0.x versiyalaridagi o'zgarishlar tufayli yuzaga keladigan mos kelmaslikni dinamik bartaraf etadi.
  2. **Routerlarni ro'yxatdan o'tkazish**:
     - `auth_router`, `admin_router`, `user_router`, `common_router`, `product_router`.
  3. **Mount operatsiyalari**:
     - `admin.mount_to(app=app)` -> `/admin` manzilida boshqaruv panelini faollashtiradi.
     - `app.mount("/static", StaticFiles(directory="static", html=True), name="static")` -> 17 ta HTML va JS/CSS fayllarini tarqatadi.
  4. **Middleware'lar**:
     - `TrustedHostMiddleware` (ruxsat berilgan hostlarni cheklash).
     - `CORSMiddleware` (front-end so'rovlariga ruxsat).
     - `SessionValidationMiddleware` (sessiya va tokenni so'rov boshida aniqlash).
     - `TimeCounter` (bajarilish vaqtini qayd etish).
     - `app.state.limiter = limiter` (SlowAPI ddos himoyasi).

### 1.3. `main.py` (Loyiha ildizida)
- Dastur paketining boshlang'ich nuqtasi (CLI yoki modul sifatida tekshirish uchun `def main(): print("Hello from market!")`).

### 1.4. `Dockerfile` & `docker-compose.yml`
- **`Dockerfile`**: `python:3.12-slim` asosida qurilgan. Zamonaviy `uv` paket menejeri orqali `pyproject.toml` faylidagi barcha kutubxonalarni keshdan foydalanib o'rnatadi.
- **`docker-compose.yml`**: 5 ta o'zaro bog'langan konteynerni bir vaqtda ishga tushiradi:
  - `market_db` (Postgres 16, port: 5432)
  - `market_redis` (Redis 7, port: 6380:6379)
  - `market_fastapi` (Uvicorn FastAPI, port: 8000)
  - `market_celery_worker` (Celery asinxron vazifalar ishchisi)
  - `market_celery_beat` (Celery rejalashtiruvchi)


In [ ]:
# Konfiguratsiya va Settings modelini tahlil qilish
from app.config import settings

print(f"Project Name: {settings.PROJECT_NAME}")
print(f"Project Version: {settings.PROJECT_VERSION}")
print(f"Algorithm: {settings.ALGORITHM}")
print(f"Access Token Expire (min): {settings.ACCESS_TOKEN_EXPIRE_MINUTES}")
print(f"Max Upload File Size: {settings.FILE_SIZE / (1024*1024)} MB")
print(f"Allowed File Types: {settings.FILE_TYPE}")


---
## 2-BOB. Ma'lumotlar Bazasi va ORM Qatlami (Database & Models)

Loyiha ma'lumotlar bazasi bilan ishlashda zamonaviy **SQLAlchemy 2.0 (Mapped / mapped_column)** deklarativ uslubidan foydalanadi.

### 2.1. `app/database.py`
- **Vazifasi**: Ma'lumotlar bazasi drayverini initsializatsiya qilish va sessiyalarni boshqarish.
- **Mexanizmi**:
  - `create_engine(DB_URL, echo=True)` -> PostgreSQL ulanish pulini (connection pool) yaratadi.
  - `SessionLocal = sessionmaker(autoflush=False, autocommit=False, bind=engine)` -> Har bir so'rov uchun alohida tranzaksion sessiya.
  - `Base(DeclarativeBase)` -> Barcha modellar meros oladigan asosiy deklarativ sinf.
  - `get_db()` generator funksiyasi:
    ```python
    def get_db():
        session = SessionLocal()
        try:
            yield session
        finally:
            session.close()
    ```
  - `db_dep = Annotated[Session, Depends(get_db)]` -> FastAPI routerlarida toza va xavfsiz sessiya olish annotatsiyasi.

### 2.2. `app/model/base.py`
- **`BaseMain`**: Barcha jadvallar uchun umumiy bo'lgan ustunlarni o'zida jamlagan abstrakt sinf (`__abstract__ = True`):
  - `id`: BigInteger, asosiy kalit (Primary Key), avtomatik oshuvchi va indekslangan.
  - `created_at`: DateTime(timezone=True), yozuv yaratilgan vaqt (`func.now()`).
  - `updated_at`: DateTime(timezone=True), har safar ma'lumot yangilanganda avtomatik yangilanuvchi vaqt (`onupdate=func.now()`).

### 2.3. Asosiy Domain Modellar:

| Fayl | Modellar | Asosiy Xususiyatlari & Biznes Mexanizmi |
| :--- | :--- | :--- |
| **`app/model/user.py`** | `User`, `User_Address`, `Courier_Profile`, `Wallet`, `Like`, `User_Rating`, `User_Search`, `UserSessionToken`, `Audit_Log`, `Role_Request` | `User` jadvali elektron pochta, xeshlangan parol, va 4 xil rol (`is_admin`, `is_manager`, `is_merchant`, `is_courier`) ga ega. Xavfsiz o'chirish (`is_deleted`, `garbage_email`) qo'llanilgan. `Role_Request` orqali kuryerlik yoki do'kon ochish arizalari ko'rib chiqiladi. `Wallet` foydalanuvchining shaxsiy hisobini saqlaydi. |
| **`app/model/product.py`** | `Shop`, `Shop_Rating`, `Media`, `Product_Media`, `Product`, `Shop_Product`, `Category`, `Comment`, `Discount` | `Shop` sotuvchi do'koni. `Shop_Rating` jadvalida `UniqueConstraint("user_id", "shop_id")` orqali har bir foydalanuvchi bitta do'konga faqat bir marta baho bera olishi kafolatlangan. `Product` o'z narxi, o'lchami, qoldig'i (`current_quantity`), muddati (`expiration_date`) va chegirmasiga (`discount_id`) ega. |
| **`app/model/order.py`** | `Order`, `Delivery_Process`, `Order_Item`, `Bucket`, `Bucket_Product`, `Promo_Code`, `Promo_Code_Report` | `Order` buyurtma yozuvlari va umumiy summasi (`total_amount`). `Delivery_Process` unikal `tracking_code` orqali kuryer yetkazib berish jarayonini (5 bosqich) nazorat qiladi. `Bucket` va `Bucket_Product` savatcha zaxirasi. `Promo_Code` chegirmalarni hisoblash uchun xizmat qiladi. |
| **`app/model/payment.py`** | `Payment_Process`, `Transaction_Log` | `Payment_Process` to'lov holati (`pending`, `completed`, `failed`). `Transaction_Log` tranzaksiya parametrlarini (`transaction_params`) JSON/matn ko'rinishida qayd etadi. |
| **`app/model/common.py`** | `Region`, `Country`, `Notification`, `User_Notification` | Hududlar ierarxiyasi (`Country` 1:M `Region` 1:M `User_Address`). `Notification` va `User_Notification` orqali tizim xabarlari belgilangan maqsadli rollarga (`ManageNotificationCreate`) yuboriladi. |

### 2.4. `alembic/` va `alembic/env.py`
- Ma'lumotlar bazasi sxemasini versiyalash. `env.py` fayli `Base.metadata` ni o'qib olib, modellardagi o'zgarishlarni avtomatik aniqlaydi (`target_metadata = Base.metadata`).


In [ ]:
# Barcha modellarning ma'lumotlar bazasi metadatasini tahlil qilish
from app.database import Base
import app.model  # barcha modellarni yuklash

print(f"Jami ro'yxatdan o'tgan jadvallar soni: {len(Base.metadata.tables)}")
print("-" * 50)
for table_name in sorted(Base.metadata.tables.keys()):
    table = Base.metadata.tables[table_name]
    pk_cols = [c.name for c in table.primary_key.columns]
    fk_cols = [f"{fk.parent.name} -> {fk.target_fullname}" for fk in table.foreign_keys]
    print(f"Jadval: {table_name:<20} | PK: {pk_cols} | FK soni: {len(fk_cols)}")


---
## 3-BOB. Xavfsizlik, Yordamchi Modullar va Fon Vazifalari (Security, Utils & Celery)

### 3.1. `app/utils.py`
Tizimning asosiy kriptografik va yordamchi funksiyalari:
1. **Parol xavfsizligi (`Argon2`)**:
   - `hash_password(password)`: `CryptContext(schemes=["argon2"])` orqali parolni xeshlaydi. Argon2 hozirgi kunda GPU va ASIC hujumlariga qarshi eng mustahkam zamonaviy standart hisoblanadi.
   - `verify_password(plain, hashed)`: Xeshni kiritilgan ochiq parol bilan taqqoslaydi.
2. **JWT Tokenlar (`python-jose`)**:
   - `generate_jwt_token(user_id, only_access=False)`:
     - `access_token`: Yaroqlilik muddati 60 daqiqa (`sub` da `user_id` saqlanadi).
     - `refresh_token`: Yaroqlilik muddati 1 kun.
   - `decode_jwt_token(token)`: Tokenni `SECRET_KEY` va `ALGORITHM` orqali tekshirib, payloadni qaytaradi. Xatolik bo'lsa `None` beradi.
3. **Pochta servisi (`send_email`)**:
   - `smtplib` orqali Gmail SMTP serveriga ulanadi (`starttls()`) va xabarlarni yuboradi.
4. **Redis aloqasi**:
   - `redis_url = redis.from_url(settings.REDIS_URL)` orqali Redis klientini boshqaradi.
5. **Enum sinflari**:
   - `Target` (WEEK, MONTH) - davriy chegirmalar uchun.
   - `ManageNotificationCreate` (SINGLE, ALL, ACTIVE_USER, MERCHANTS, COURIERS, MANAGERS) - xabarnoma auditoriyasi.
   - `ChooseRoleRequest` (MERCHANT, COURIER, MANAGER, ADMIN) - kuryerlik/do'kon ochish rollari.
   - `RoleRequestStatus` (PENDING, APPROVED, REJECTED, WAITING).

### 3.2. `app/middleware.py`
Har bir HTTP so'rov routerga yetib borguncha ushbu filtrlar orqali o'tadi:
1. **`SessionValidationMiddleware`**:
   - Request Header'dan `Authorization: Bearer <token>` ni yoki Cookie'dagi `access_token` ni qidiradi.
   - Tokenni dekod qilib, uning amal qilish muddatini (`exp`) Unix vaqti bilan solishtiradi.
   - Muvaffaqiyatli bo'lsa, `request.state.user = token_payload` qilib belgilaydi.
   - `X-Middleware-time` javob sarlavhasini o'rnatadi.
2. **`TimeCounter`**:
   - So'rovning umumiy bajarilish vaqtini (`perf_counter`) o'lchab, `X-proccess-time` sarlavhasida qaytaradi.
3. **`limiter` (SlowAPI)**:
   - IP manzil bo'yicha so'rovlar chastotasini cheklaydi (Brute-force va DDOS hujumlarining oldini oladi).

### 3.3. `app/dependense.py`
FastAPI dependency inyeksiya mexanizmlari:
- **`get_current_active_user`**: `request.state.user` yoki `jwt_securty` bearer tokenidan foydalanuvchini bazadan oladi. Agar user topilmasa, o'chirilgan bo'lsa (`is_deleted=True`) yoki nofaol bo'lsa (`is_active=False`), `401 Unauthorized` xatosini beradi.
- **RBAC Huquqlari**:
  - `get_current_login_user`: Admin, Manager, Courier yoki Merchant ekanligini tasdiqlaydi.
  - `get_current_courier_user`: Faqat kuryerlar uchun (`is_courier=True`).
  - `get_current_merchant_user`: Faqat do'kon egalari uchun (`is_merchant=True`).
  - `get_current_manager_user`: Faqat menejerlar uchun (`is_manager=True`).
  - `get_current_admin_user`: Faqat administratorlar uchun (`is_admin=True`).
- **Yordamchi filterlar**:
  - `get_pagination`: `min` va `max` parametrlarini oladi.
  - `current_discount_date`: Hafta yoki oy orqaga hisoblab chegirma sanasi filtrini beradi.
  - `get_current_product`: Mahsulot mavjudligi va faolligini tekshiradi.

### 3.4. `app/celery.py`
- Celery obyekti (`market_tasks`).
- `@celery.task(name="send_email_message")`: Email yuborish operatsiyasi og'ir tarmoq I/O amali bo'lgani sababli, Uvicorn asosiy oqimini (event loop) bloklamasligi uchun Redis orqali Celery ishchi xodimiga (worker) topshiriladi.


In [ ]:
# Xavfsizlik: Parollash va JWT mexanizmini amalda sinash
from app.utils import hash_password, verify_password, generate_jwt_token, decode_jwt_token

test_password = "SuperSecret_Password_2026!"
hashed = hash_password(test_password)
print(f"Argon2 Xesh: {hashed}")
print(f"To'g'ri parol tekshiruvi: {verify_password(test_password, hashed)}")
print(f"Noto'g'ri parol tekshiruvi: {verify_password('wrong_password', hashed)}")

# JWT yaratish va dekodlash
access_tok, refresh_tok = generate_jwt_token(user_id=42)
print("-" * 50)
print(f"Access Token (boshi): {access_tok[:35]}...")
print(f"Refresh Token (boshi): {refresh_tok[:35]}...")

decoded = decode_jwt_token(access_tok)
print(f"Dekodlangan Payload: {decoded}")


---
## 4-BOB. Ma'lumotlar Validatsiyasi va Sxemalar (Pydantic Schemas)

`app/schemas/` papkasidagi DTO (Data Transfer Objects) modellari kiruvchi ma'lumotlarni qat'iy tekshiradi va chiquvchi javoblarni formatlaydi:

### 4.1. `app/schemas/base.py`
- `Base(BaseModel)`: `model_config = ConfigDict(from_attributes=True)` sozlamasiga ega. Bu SQLAlchemy ORM obyektlarini to'g'ridan to'g'ri Pydantic orqali JSON ko'rinishiga o'girish imkonini beradi.

### 4.2. `app/schemas/auth.py`
- **`UserRegisterRequest`**:
  - `email`: `EmailStr` formati.
  - `password`, `password2`: Ikki marta kiritilgan parollar mosligini tekshiradi.
  - `@model_validator(mode="after")` orqali:
    1. `password == password2` mosligini tekshiradi.
    2. Parol uzunligi kamida 8 ta belgi bo'lishini talab qiladi.
    3. `zxcvbn(password)["score"] >= 2` talabini qo'yadi. Agar parol juda oddiy bo'lsa (masalan, "password123"), xatolik beradi.
- **`UserRegisterResponse`**: Foydalanuvchi ID, email va yaratilgan sanasini qaytaradi.
- **`UserRegisterRoleRequest` & `UserRegisterRoleResponse`**: Kuryer yoki merchant bo'lish arizasi sxemasi.
- **`AdminMarkedRoleRequest` & `AdminMarkedRoleResponse`**: Administrator arizani ko'rib chiqqanligini tasdiqlovchi sxemalar.

### 4.3. `app/schemas/product.py` & `app/schemas/home.py`
- **`OneProductDetailsRespones`**: Mahsulotning to'liq kartochkasi:
  - `id`: int (gt=0)
  - `name`: str (min_length=2)
  - `product_size`: Decimal
  - `product_rating_avg`: Decimal (0.0 dan 5.0 gacha)
  - `price`: Decimal (chegirma hisoblangan haqiqiy narx)
  - `is_liked`: bool (joriy xaridor layk bosganmi yoki yo'q)
  - `users_comments`, `comment_count`, `discount_title`.
- **`ProductListResponse`**: Bosh sahifa va qidiruv natijalaridagi mahsulotlar qisqa ro'yxati.


In [ ]:
# Pydantic va zxcvbn parol validatsiyasini sinash
from app.schemas.auth import UserRegisterRequest
from pydantic import ValidationError

# 1. Zaif parol bilan sinov
try:
    UserRegisterRequest(email="test@market.uz", password="123", password2="123")
except ValidationError as e:
    print("Kutilgan xatolik (zaif parol):", e.errors()[0]['msg'])

# 2. Kuchli parol bilan sinov
valid_user = UserRegisterRequest(
    email="test@market.uz",
    password="KuchliParol_2026_Compl3x!",
    password2="KuchliParol_2026_Compl3x!"
)
print("Muvaffaqiyatli validatsiya:", valid_user.email)


---
## 5-BOB. API Marshrutizatsiyasi va Biznes Mantiq (API v1 Routers)

Loyihaning barcha REST endpointlari `/api/v1` prefiksi ostida jamlangan:

### 5.1. Autentifikatsiya: `app/api/v1/auth/`
- **`register.py`**:
  - **`POST /api/v1/user/auth/register`**:
    1. Email allaqachon mavjud va faol ekanligini tekshiradi (`User.is_active == True`).
    2. Yangi foydalanuvchi ma'lumotlarini parolini xeshlagan holda `user_temp` lug'atiga joylaydi.
    3. `secrets.token_hex(10)` orqali 20 belgili maxfiy OTP kod yaratadi.
    4. Redis bazasida `redis_url.setex(secret_kod, 300, json.dumps(user_temp))` orqali 5 daqiqaga (300 soniya) saqlaydi.
    5. `send_email_message.delay(...)` orqali Celery vazifasini ishga tushirib, pochtaga kodni asinxron yuboradi.
  - **`POST /api/v1/user/auth/verify/{secret_code}`**:
    1. Kiritilgan kod bo'yicha Redisdan ma'lumotni qidiradi.
    2. Agar ma'lumot topilsa, foydalanuvchini `User` jadvaliga yozadi (`is_active=True`).
    3. **Muhim biznes qoida**: Agar bazada hali birorta ham user bo'lmasa, birinchi ro'yxatdan o'tgan foydalanuvchiga avtomatik ravishda `is_admin = True` va `is_manager = True` maqomi beriladi!
    4. Foydalanuvchiga JWT access va refresh tokenlarni yaratadi va xavfsiz `HttpOnly`, `SameSite=lax` Cookie'ga joylaydi.
    5. Redisdagi bir martalik OTP kodni o'chiradi.
  - **`POST /api/v1/user/auth/register/role`**:
    - Faol foydalanuvchilar kuryer (`courier`) yoki sotuvchi (`merchant`) bo'lish uchun rezyume fayli (`UploadFile`) va ariza matnini topshiradi.
  - **`POST /api/v1/user/auth/refresh_tokens`**:
    - Cookiedagi refresh tokenni tekshirib, yangi access token ishlab beradi.
- **`gwt.py`**:
  - `POST /api/v1/jwt/login`: To'g'ridan to'g'ri JSON orqali email va parolni tekshirib, tokenlar juftligini qaytaradi.
  - `POST /api/v1/jwt/update_access_token`: Body orqali tokenni yangilash.
  - `GET /api/v1/jwt/profile`: `current_user_jwt_dep` orqali foydalanuvchi profilini qaytaradi.

### 5.2. Foydalanuvchi va Hamyon: `app/api/v1/user/`
- **`user.py`**:
  - `GET /api/v1/user/detail/{user_id}`: Foydalanuvchi tafsilotlari.
  - `GET /api/v1/user/get_user_address/{user_id}`: `options(joinedload(User_Address.region).joinedload(Region.country))` orqali bitta SQL so'rovida foydalanuvchi manzili, viloyati va davlatini xotiraga yuklaydi (N+1 muammosini oldini oladi).
  - `POST /api/v1/user/create/like`: Mahsulotga layk bosish.
  - `DELETE /api/v1/user/delete-user/{user_id}`: Foydalanuvchini o'chirish.
- **`wallet.py`**:
  - `GET /api/v1/user/wallet/balance/{user_id}`: Foydalanuvchining hamyon balansi va bloklanmaganligini (`is_blocked=False`) tekshiradi.

### 5.3. Mahsulotlar va Media: `app/api/v1/product/`
- **`product.py`**:
  - `GET /api/v1/product/details/{product_id}`: Loyihadagi **eng murakkab va optimallashtirilgan SQL so'rovlaridan biri**:
    - Bitta so'rovda `Product`, `User_Rating`, `Product_Media`, `Category`, `Comment`, `Discount` jadvallarini `outerjoin` qiladi.
    - `case` va `coalesce` orqali chegirma faol bo'lsa, narxni avtomatik kamaytirib beradi: `price - (price * percent / 100)`.
    - Reyting o'rtachasi (`func.avg(User_Rating.ball)`) va sharhlar sonini hisoblaydi.
    - `exists().where(Like.product_id == Product.id, Like.user_id == user.id)` orqali joriy xaridor ushbu mahsulotga layk bosgan-bosmaganini aniqlaydi.
  - `GET /api/v1/product/search-by-category`, `GET /api/v1/product/search-by-name`: Qidiruv mexanizmlari.
- **`media.py`**:
  - `POST /api/v1/media/uploadfile/`: Fayl hajmini (max 5MB) va kengaytmasini tekshiradi, `media/` jildiga saqlaydi va bazada `Media` yozuvini yaratadi.

### 5.4. Umumiy Modullar (Home, Notifications, Location): `app/api/v1/common/`
- **`home.py`**:
  - `GET /api/v1/home/discount-products`: Mavsumiy faol chegirmali tovarlar.
  - `GET /api/v1/home/monthly-discount`: Oxirgi oy yoki hafta ichida chegirmaga tushgan mahsulotlar.
  - `GET /api/v1/home/top-10-products`: Qidiruvlar soni (`User_Search`) va reytingi eng baland tovarlar.
  - `GET /api/v1/home/liked-products`: Foydalanuvchining shaxsiy sevimlilar ro'yxati.
- **`notification.py`**: Xabarnomalar ro'yxati va ularni o'qish.
- **`location.py`**: Region va Country boshqaruvi.

### 5.5. Hozirda Tayyorlanayotgan Qismlar (Roadmap / Empty Modules)
- `app/api/v1/order/`: `bucked.py`, `order.py`, `deliver.py`, `promo_code.py` (kelgusida savatchadan xarid qilish va kuryer tracking jarayonlari to'ldiriladi).
- `app/api/v1/payment/`: `payment.py`, `transaction.py` (to'lov tizimlari shlyuzi).
- `app/api/v1/user/courier.py`: Kuryerlik koordinatalari va yetkazib berish topshiriqlari.


In [ ]:
# FastAPI Ilovasidagi barcha faol marshrutlar (Routes) ro'yxatini chiqarish
from app.main import app

routes_info = []
for route in app.routes:
    methods = getattr(route, "methods", None)
    if methods:
        routes_info.append({
            "path": route.path,
            "name": route.name,
            "methods": list(methods)
        })

print(f"Jami ro'yxatdan o'tgan endpointlar soni: {len(routes_info)}")
print("-" * 70)
for r in sorted(routes_info, key=lambda x: x["path"])[:15]:
    print(f"{r['methods'][0]:<7} | {r['path']}")


---
## 6-BOB. Starlette Admin Boshqaruv Tizimi (`app/admin/`)

Admin panel ma'lumotlar bazasini qulay veb-interfeys orqali boshqarish uchun xizmat qiladi:

### 6.1. `app/admin/settings.py`
- `admin = Admin(engine, title="Market-place admin", base_url="/admin", auth_provider=JsonAuthProvider(...))`
- Quyidagi asosiy ko'rinishlar ro'yxatdan o'tkazilgan:
  - `UserAdminView(User, icon="fa fa-user")`
  - `NotificationAdminView(Notification)`
  - `RoleRequestView(Role_Request)`

### 6.2. `app/admin/auth.py`
- **`JsonAuthProvider`**:
  - Admin panelga kirishda xavfsizlikni ta'minlaydi.
  - `@limiter.limit("10/minute")`: Minutiga 10 martadan ortiq login urinishlariga ruxsat bermaydi.
  - Foydalanuvchi faol bo'lishi (`is_active=True`, `is_deleted=False`) va **albatta** `is_admin`, `is_manager` yoki `is_merchant` huquqlaridan biriga ega bo'lishi shart! Oddiy xaridorlar admin panelga kira olmaydi.

### 6.3. `app/admin/views.py` & `app/admin/action.py`
- **`UserAdminView`**:
  - Maxfiy maydonlarni (`password_hash`, `bio`, `last_login`, `is_deleted`) ro'yxatdan yashiradi.
  - `before_create` va `before_edit` hooklari orqali admindan yangi parol kiritilsa, uni avtomatik `hash_password(...)` orqali xeshlab bazaga yozadi.
- **`RoleRequestView` va `action.py`**:
  - `/admin/get/role-aplications`: Barcha hamkorlik arizalarini ko'rish.
  - `/admin/marked/role`: Administrator arizani ko'rib chiqqach, uning statusini `WAITING` ga o'tkazib, 1 kunlik ko'rib chiqish muddati (`status_expired_at`) va ko'rib chiqqan admin ID sini biriktiradi.


---
## 7-BOB. Frontend Qatlami va Integratsiya (`static/`)

Frontend toza Vanilla JavaScript va zamonaviy CSS Custom Properties asosida yaratilgan bo'lib, quyidagi 17 ta sahifadan iborat:

### 7.1. Klient Xizmat Modullari
- **`static/js/api.js`**: `MarketAPI` sinfi. Hozirgi bosqichda brauzer `localStorage` xotirasi bilan ishlaydi (`market_cart`, `market_likes`, `market_orders`, `market_courier_tasks`). Har bir metod ichida backend API ga to'liq ulanish uchun `fetch(API_BASE_URL + ...)` shablonlari tayyorlab qo'yilgan.
- **`static/js/app.js`**:
  - `formatPrice(amount)`: Pul miqdorlarini o'zbek so'mi formatida chiqarish (`12 000 000 so'm`).
  - `showToast(message, type)`: Animatsiyali bildirishnomalar (`success`, `warning`, `danger`, `info`).
  - `updateHeaderCounters()`: Headerdagi savatdagi tovarlar soni va layklar sonini barcha sahifalarda bir xil yangilab turish.
- **`static/js/mock-data.js`**: Frontend sahifalarini oflayn sinash uchun sinov ma'lumotlari.

### 7.2. 17 ta Sahifa va Ularning Backend Bog'liqlik Xaritasi

| Sahifa Fayli | UI Vazifasi | Bog'langan Backend Endpointi |
| :--- | :--- | :--- |
| `home-page.html` | Bosh sahifa, aksiyalar va tavsiyalar | `GET /api/v1/home/discount-products`, `top-10-products` |
| `one-product-detail.html` | Mahsulot kartasi va tavsifi | `GET /api/v1/product/details/{id}`, `POST /user/create/like` |
| `category-page.html` | Toifalar daraxti | `GET /api/v1/product/search-by-category` |
| `product-by-this-type.html` | Kategoriya bo'yicha grid | `GET /api/v1/product/search-by-name` |
| `bucket-page.html` | Savatcha va tanlangan tovarlar | `POST /api/v1/order/push-product` (ishlab chiqilmoqda) |
| `order-procces-page.html` | Buyurtmani rasmiylashtirish | `POST /api/v1/order/create` (ishlab chiqilmoqda) |
| `ordered-product-page.html` | Buyurtmalar tarixi | `GET /api/v1/order/my-orders` |
| `order-tracking-page.html` | Kuryerni 5 bosqichda kuzatish | `GET /api/v1/order/tracking/{code}` |
| `courier-page.html` | Kuryer ish portali (Uzum Tezkor analogi) | `GET /api/v1/courier/tasks` |
| `account-page.html` | Shaxsiy profil va sozlamalar | `GET /api/v1/jwt/profile`, `PUT /user/profile` |
| `wallet-page.html` | Hamyon va to'lov kartalari | `GET /api/v1/user/wallet/balance/{id}` |
| `shop-detail-page.html` | Do'kon vitrinasi va banneri | `GET /api/v1/product/shop/{id}` |
| `notifications-page.html` | Xabarnomalar markazi | `GET /api/v1/notifications/user-notifications` |
| `role-request-page.html` | Kuryer/Merchant arizalari | `POST /api/v1/user/auth/register/role` |
| `liked-page.html` | Sevimlilar to'plami | `GET /api/v1/home/liked-products` |
| `one-product-comments.html` | Xaridorlar sharhlari | `GET /api/v1/product/comments/{id}` |
| `chatting-with-buyer.html` | Xaridor va kuryer/do'kon chati | WebSocket / Polling |


---
## 8-BOB. `TZ.md` (44 ta Vazifa) va Backend Rivojlantirish Yo'l Xaritasi

Loyiha texnik topshirig'i (`TZ.md`) 5 ta katta yo'nalish bo'yicha 44 ta masalani o'z ichiga olgan:

### 8.1. Hozirda Qamrab Olingan Vazifalar:
- **1-Topshiriq**: Faol foydalanuvchilar ro'yxati (`GET /api/v1/user/get_users?is_active=True`).
- **2-Topshiriq**: Foydalanuvchining manzillarini bog'langan region va country bilan olish (`joinedload`).
- **4-Topshiriq**: Admin va menejerlarni aniqlash (RBAC va Starlette Admin filtri).
- **21-Topshiriq**: Faol mahsulotlar nomi va narxlarini chiqarish (`GET /api/v1/home/discount-products`).
- **22-Topshiriq**: Muayyan kategoriya mahsulotlari (`GET /api/v1/product/search-by-category`).
- **28-Topshiriq**: Mahsulotga layk bosgan foydalanuvchilar (`Like` jadvali).
- **30-Topshiriq**: Nomi bo'yicha mahsulot qidirish (`GET /api/v1/product/search-by-name`).
- **33-Topshiriq**: Foydalanuvchi hamyon balansi (`GET /api/v1/user/wallet/balance/{user_id}`).
- **41-Topshiriq (qisman)**: Mahsulot chegirmasi bo'yicha yakuniy narxni hisoblash (SQL `case` va `coalesce` orqali `Product.price - (price * percent / 100)`).

### 8.2. Keyingi Bosqichda Bajarilishi Rejalashtirilgan Vazifalar (`GEMINI_adice.md` bo'yicha):
1. **Savatcha va Zaxira Mantiqi (43-Topshiriq)**:
   - Xaridor savatga mahsulot qo'shayotganda `SELECT ... FOR UPDATE` blokirovkasi orqali ombordagi qoldiqni tekshirish va bir vaqtning o'zida bir nechta xarid bo'lganda poyga holatini (Race Condition) oldini olish.
2. **Kuryer va Tracking Tizimi (42-Topshiriq)**:
   - Kuryer buyurtmani `yetkazildi` holatiga o'tkazganda, buyurtma statusini avtomatik `Yakunlandi` holatiga o'tkazish.
3. **Audit Log Middleware**:
   - Har bir `POST`, `PUT`, `DELETE` amallarini `Audit_Log` jadvaliga yozib boruvchi middleware yaratish.


---
## 9-BOB. Xulosa va Arxitektura Bahosi

`market` loyihasi o'zbek elektron tijorat bozori uchun juda puxta o'ylangan va professional backend arxitekturasiga ega:
1. **Asinxron va Yuqori Tezlik**: FastAPI, Redis va Celery orqali yuklamalar teng taqsimlangan.
2. **Xavfsizlik**: Zamonaviy Argon2 shifrlash, JWT sessiyalari, OTP orqali email tasdiqlash va SlowAPI rate limiting to'liq joriy etilgan.
3. **ORM Optimallashtirish**: N+1 muammolari `joinedload` va `selectinload` orqali oldi olingan; mahsulot kartochkasi murakkab outerjoin agregatlari orqali bitta SQL so'rovida qaytariladi.
4. **Modullilik**: Har bir domen (auth, user, product, order, payment, common, admin) o'zining model, sxema va routerlariga ega bo'lib, loyihani kengaytirish va yangi funksiyalarni qo'shish juda oson.
